In [ ]:
from pathlib import Path

ROOT = Path(".").resolve().parents[1]
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
from rich import print as rprint

In [ ]:
from dotenv import load_dotenv

load_dotenv("../.env")

In [ ]:
from src.application.contracts import PipelineRequest
from src.application import ViRAGEPipeline
from src.application.settings import ViRAGESettings

tmp_path = Path("../demo_data/tmp_folder")
data_path = Path("../demo_data/Iris.csv")
settings = ViRAGESettings(artifact_root=tmp_path / "artifacts")
pipeline = ViRAGEPipeline(settings)
query="Show the sales trend over time"
request = PipelineRequest(query=query, data_path=data_path.as_posix())

In [ ]:
from langchain_ollama import ChatOllama

LLM_MODEL = "gemma3:1b"  # или "llama3.2:1b"
# LLM_MODEL = "llama3.2:1b"       # или "llama3.2:1b"
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
)

In [ ]:
from src.infrastructure import RuntimeContext

runtime = RuntimeContext(settings=settings, llm=llm)

In [ ]:
from src.services import QueryUnderstandingService

qu = QueryUnderstandingService().invoke(runtime=runtime, user_context=request.user_context, query=request.query)

In [ ]:
rprint("query:", request.query)
rprint(qu)

In [ ]:
from src.services import CanonicalPlanningService

cp = CanonicalPlanningService().invoke(runtime=runtime, query_understanding=qu)

In [ ]:
rprint(cp)

In [ ]:
from src.services import DataProfilerService

data_profile = DataProfilerService().invoke(runtime=runtime, data_path=data_path)

In [ ]:
rprint(data_profile)

In [ ]:
from src.services import DataPreparationService

data_prep = DataPreparationService().invoke(runtime=runtime, data_path=data_path, data_profile=data_profile, run_id="1")

In [ ]:
rprint(data_prep)

# VisRAG

In [ ]:
from src.services import VisRAGService

recommendations = VisRAGService().invoke(runtime=runtime, data_profile=data_profile, query_understanding=qu)

In [ ]:
rprint(recommendations)